##Simulated Dataset

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
import random
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

items = [
    ("Chicken Bowl", "Main", 12.5),
    ("Paneer Bowl", "Main", 11.0),
    ("Veg Wrap", "Snack", 8.0),
    ("Fries", "Snack", 4.5),
    ("Burger", "Main", 9.5),
    ("Pizza Slice", "Main", 10.0),
    ("Salad", "Healthy", 7.5),
    ("Smoothie", "Drink", 6.0)
]

cities = ["Los Angeles", "San Francisco", "San Diego"]
order_modes = ["Online", "In-Store"]

data = []
start_date = datetime(2026, 4, 1)

for i in range(1, 100001):
    item = random.choice(items)
    quantity = random.randint(1, 4)
    order_time = start_date + timedelta(
        days=random.randint(0, 10),
        hours=random.randint(9, 22),
        minutes=random.randint(0, 59)
    )

    data.append((
        str(1000 + i),
        item[0],
        item[1],
        quantity,
        item[2],
        order_time.strftime("%Y-%m-%d %H:%M:%S"),
        random.choice(order_modes),
        random.choice(cities),
        random.choice([0, 1])
    ))

columns = [
    "order_id",
    "item_name",
    "item_type",
    "quantity",
    "price",
    "order_time",
    "order_mode",
    "city",
    "customization_flag"
]

df_orders_raw = spark.createDataFrame(data, columns)
df_orders_raw = df_orders_raw.withColumn("order_time", to_timestamp(col("order_time")))

df_orders_raw.show(10, truncate=False)
df_orders_raw.printSchema()

+--------+------------+---------+--------+-----+-------------------+----------+-------------+------------------+
|order_id|item_name   |item_type|quantity|price|order_time         |order_mode|city         |customization_flag|
+--------+------------+---------+--------+-----+-------------------+----------+-------------+------------------+
|1001    |Fries       |Snack    |3       |4.5  |2026-04-11 21:46:00|In-Store  |Los Angeles  |0                 |
|1002    |Burger      |Main     |4       |9.5  |2026-04-04 21:48:00|Online    |Los Angeles  |1                 |
|1003    |Fries       |Snack    |1       |4.5  |2026-04-04 11:05:00|In-Store  |San Diego    |1                 |
|1004    |Veg Wrap    |Snack    |1       |8.0  |2026-04-07 10:41:00|In-Store  |San Francisco|1                 |
|1005    |Chicken Bowl|Main     |4       |12.5 |2026-04-06 14:09:00|In-Store  |San Francisco|1                 |
|1006    |Salad       |Healthy  |1       |7.5  |2026-04-03 18:08:00|In-Store  |San Francisco|1  

## Bronze Layer: Raw Data Ingestion
The raw restaurant order dataset is stored in Delta format to simulate the ingestion layer of a lakehouse architecture.

In [0]:
# Bronze Layer: Raw Data Ingestion

# Save raw data as a Delta table
df_orders_raw.write.mode("overwrite").format("delta").saveAsTable("restaurant_bronze")

# Read Bronze table
df_bronze = spark.table("restaurant_bronze")

# Preview data
df_bronze.show(10, truncate=False)
df_bronze.printSchema()

+--------+------------+---------+--------+-----+-------------------+----------+-----------+------------------+
|order_id|item_name   |item_type|quantity|price|order_time         |order_mode|city       |customization_flag|
+--------+------------+---------+--------+-----+-------------------+----------+-----------+------------------+
|88501   |Smoothie    |Drink    |4       |6.0  |2026-04-02 17:11:00|In-Store  |San Diego  |0                 |
|88502   |Veg Wrap    |Snack    |3       |8.0  |2026-04-03 16:20:00|In-Store  |San Diego  |0                 |
|88503   |Chicken Bowl|Main     |3       |12.5 |2026-04-03 19:49:00|Online    |San Diego  |0                 |
|88504   |Smoothie    |Drink    |4       |6.0  |2026-04-08 13:43:00|Online    |San Diego  |0                 |
|88505   |Veg Wrap    |Snack    |4       |8.0  |2026-04-09 21:15:00|Online    |San Diego  |1                 |
|88506   |Veg Wrap    |Snack    |4       |8.0  |2026-04-06 22:32:00|Online    |San Diego  |0                 |
|

## Silver Layer: Cleaning and Feature Engineering
In this layer, the raw data is cleaned, standardized, and enriched with derived columns for analysis.

In [0]:
from pyspark.sql.functions import col, hour, dayofweek, when

# Read Bronze table
df_bronze = spark.table("restaurant_bronze")

# Clean data
df_clean = df_bronze.dropna().dropDuplicates()

# Ensure correct data types
df_clean = df_clean \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("price", col("price").cast("double"))

# Feature engineering
df_silver = df_clean \
    .withColumn("total_amount", col("quantity") * col("price")) \
    .withColumn("order_hour", hour(col("order_time"))) \
    .withColumn("order_day", dayofweek(col("order_time"))) \
    .withColumn(
        "time_category",
        when(col("order_hour").between(9, 12), "Morning")
        .when(col("order_hour").between(13, 17), "Afternoon")
        .otherwise("Evening")
    ) \
    .withColumn(
        "customization_label",
        when(col("customization_flag") == 1, "Customized").otherwise("Standard")
    )

# Save Silver as Delta table
df_silver.write.mode("overwrite").format("delta").saveAsTable("restaurant_silver")

# Read Silver table
df_silver = spark.table("restaurant_silver")

# Preview
df_silver.show(10, truncate=False)
df_silver.printSchema()

+--------+------------+---------+--------+-----+-------------------+----------+-----------+------------------+------------+----------+---------+-------------+-------------------+
|order_id|item_name   |item_type|quantity|price|order_time         |order_mode|city       |customization_flag|total_amount|order_hour|order_day|time_category|customization_label|
+--------+------------+---------+--------+-----+-------------------+----------+-----------+------------------+------------+----------+---------+-------------+-------------------+
|1012    |Pizza Slice |Main     |4       |10.0 |2026-04-01 17:24:00|Online    |San Diego  |1                 |40.0        |17        |4        |Afternoon    |Customized         |
|1019    |Fries       |Snack    |3       |4.5  |2026-04-04 17:55:00|In-Store  |Los Angeles|1                 |13.5        |17        |7        |Afternoon    |Customized         |
|1028    |Salad       |Healthy  |1       |7.5  |2026-04-06 21:28:00|Online    |San Diego  |1             

## Gold Layer: Business Aggregations
This layer creates analytical tables that summarize revenue, customer preferences, and restaurant order behavior.

##A. Revenue by city

In [0]:
df_revenue_city = df_silver.groupBy("city") \
    .sum("total_amount") \
    .withColumnRenamed("sum(total_amount)", "total_revenue") \
    .orderBy(col("total_revenue").desc())

df_revenue_city.show()

+-------------+-------------+
|         city|total_revenue|
+-------------+-------------+
|San Francisco|     723371.0|
|  Los Angeles|     714276.0|
|    San Diego|     714039.5|
+-------------+-------------+



##B. Top-selling items by quantity

In [0]:
from pyspark.sql.functions import sum as spark_sum

df_top_items = df_silver.groupBy("item_name") \
    .agg(
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_sum("total_amount").alias("total_revenue")
    ) \
    .orderBy(col("total_quantity_sold").desc())

df_top_items.show()

+------------+-------------------+-------------+
|   item_name|total_quantity_sold|total_revenue|
+------------+-------------------+-------------+
|       Salad|              31625|     237187.5|
|    Smoothie|              31618|     189708.0|
|    Veg Wrap|              31583|     252664.0|
|Chicken Bowl|              31136|     389200.0|
| Paneer Bowl|              31130|     342430.0|
|       Fries|              31060|     139770.0|
| Pizza Slice|              30902|     309020.0|
|      Burger|              30706|     291707.0|
+------------+-------------------+-------------+



##C. Average order value by order mode

In [0]:
from pyspark.sql.functions import avg

df_avg_order_mode = df_silver.groupBy("order_mode") \
    .agg(avg("total_amount").alias("avg_order_value")) \
    .orderBy(col("avg_order_value").desc())

df_avg_order_mode.show()

+----------+------------------+
|order_mode|   avg_order_value|
+----------+------------------+
|    Online|21.614518072289158|
|  In-Store|21.419990039840638|
+----------+------------------+



##D. Peak ordering hours

In [0]:
from pyspark.sql.functions import count

df_peak_hours = df_silver.groupBy("order_hour") \
    .agg(count("*").alias("order_count")) \
    .orderBy(col("order_count").desc())

df_peak_hours.show()

+----------+-----------+
|order_hour|order_count|
+----------+-----------+
|        14|       7293|
|        15|       7264|
|        16|       7205|
|        22|       7170|
|        19|       7166|
|        10|       7161|
|        18|       7138|
|        13|       7114|
|         9|       7109|
|        17|       7094|
|        12|       7082|
|        20|       7074|
|        21|       7071|
|        11|       7059|
+----------+-----------+



##E. Customization behavior

In [0]:
df_customization = df_silver.groupBy("customization_label") \
    .agg(
        count("*").alias("num_orders"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .orderBy(col("num_orders").desc())

df_customization.show()

+-------------------+----------+------------------+
|customization_label|num_orders|   avg_order_value|
+-------------------+----------+------------------+
|           Standard|     50181|21.537464378948208|
|         Customized|     49819| 21.49611593970172|
+-------------------+----------+------------------+



##F. Daily Revenue Trend Analysis

In [0]:
from pyspark.sql.functions import to_date, sum as spark_sum, count

df_daily_revenue = df_silver \
    .withColumn("order_date", to_date(col("order_time"))) \
    .groupBy("order_date") \
    .agg(
        spark_sum("total_amount").alias("daily_revenue"),
        count("*").alias("total_orders")
    ) \
    .orderBy("order_date")

df_daily_revenue.show()

+----------+-------------+------------+
|order_date|daily_revenue|total_orders|
+----------+-------------+------------+
|2026-04-01|     196820.5|        9138|
|2026-04-02|     193146.0|        9038|
|2026-04-03|     199984.0|        9265|
|2026-04-04|     195535.5|        9012|
|2026-04-05|     195199.0|        9127|
|2026-04-06|     197439.5|        9276|
|2026-04-07|     196390.0|        9117|
|2026-04-08|     196116.5|        9101|
|2026-04-09|     191150.0|        8992|
|2026-04-10|     195412.0|        8993|
|2026-04-11|     194493.5|        8941|
+----------+-------------+------------+



##G. Gold tables save

In [0]:
# Save Gold tables as Delta tables 

df_revenue_city.write.mode("overwrite").format("delta").saveAsTable("restaurant_gold_revenue_city")

df_top_items.write.mode("overwrite").format("delta").saveAsTable("restaurant_gold_top_items")

df_avg_order_mode.write.mode("overwrite").format("delta").saveAsTable("restaurant_gold_avg_order_mode")

df_peak_hours.write.mode("overwrite").format("delta").saveAsTable("restaurant_gold_peak_hours")

df_customization.write.mode("overwrite").format("delta").saveAsTable("restaurant_gold_customization")

df_daily_revenue.write.mode("overwrite").format("delta").saveAsTable("restaurant_gold_daily_revenue")

##Spark SQL layer

In [0]:
#Register silver data as a temp view:
df_silver.createOrReplaceTempView("restaurant_orders")

##Query 1: Revenue by city


In [0]:
spark.sql("""
SELECT city, ROUND(SUM(total_amount), 2) AS total_revenue
FROM restaurant_orders
GROUP BY city
ORDER BY total_revenue DESC
""").show()

+-------------+-------------+
|         city|total_revenue|
+-------------+-------------+
|San Francisco|     723371.0|
|  Los Angeles|     714276.0|
|    San Diego|     714039.5|
+-------------+-------------+



##Query 2: Most popular items

In [0]:
spark.sql("""
SELECT item_name, SUM(quantity) AS total_quantity_sold
FROM restaurant_orders
GROUP BY item_name
ORDER BY total_quantity_sold DESC
""").show()

+------------+-------------------+
|   item_name|total_quantity_sold|
+------------+-------------------+
|       Salad|              31625|
|    Smoothie|              31618|
|    Veg Wrap|              31583|
|Chicken Bowl|              31136|
| Paneer Bowl|              31130|
|       Fries|              31060|
| Pizza Slice|              30902|
|      Burger|              30706|
+------------+-------------------+



##Query 3: Peak ordering times

In [0]:
spark.sql("""
SELECT order_hour, COUNT(*) AS order_count
FROM restaurant_orders
GROUP BY order_hour
ORDER BY order_count DESC
""").show()

+----------+-----------+
|order_hour|order_count|
+----------+-----------+
|        14|       7293|
|        15|       7264|
|        16|       7205|
|        22|       7170|
|        19|       7166|
|        10|       7161|
|        18|       7138|
|        13|       7114|
|         9|       7109|
|        17|       7094|
|        12|       7082|
|        20|       7074|
|        21|       7071|
|        11|       7059|
+----------+-----------+



##Query 4: Average order value by mode

In [0]:
spark.sql("""
SELECT order_mode, ROUND(AVG(total_amount), 2) AS avg_order_value
FROM restaurant_orders
GROUP BY order_mode
ORDER BY avg_order_value DESC
""").show()

+----------+---------------+
|order_mode|avg_order_value|
+----------+---------------+
|    Online|          21.61|
|  In-Store|          21.42|
+----------+---------------+



##Query 5: Customization impact

In [0]:
spark.sql("""
SELECT customization_label, 
       COUNT(*) AS num_orders,
       ROUND(AVG(total_amount), 2) AS avg_order_value
FROM restaurant_orders
GROUP BY customization_label
ORDER BY num_orders DESC
""").show()

+-------------------+----------+---------------+
|customization_label|num_orders|avg_order_value|
+-------------------+----------+---------------+
|           Standard|     50181|          21.54|
|         Customized|     49819|           21.5|
+-------------------+----------+---------------+



## Query 6: Daily revenue trend

In [0]:
spark.sql("""
SELECT 
    DATE(order_time) AS order_date,
    ROUND(SUM(total_amount), 2) AS daily_revenue,
    COUNT(*) AS total_orders
FROM restaurant_orders
GROUP BY DATE(order_time)
ORDER BY order_date
""").show()

+----------+-------------+------------+
|order_date|daily_revenue|total_orders|
+----------+-------------+------------+
|2026-04-01|     196820.5|        9138|
|2026-04-02|     193146.0|        9038|
|2026-04-03|     199984.0|        9265|
|2026-04-04|     195535.5|        9012|
|2026-04-05|     195199.0|        9127|
|2026-04-06|     197439.5|        9276|
|2026-04-07|     196390.0|        9117|
|2026-04-08|     196116.5|        9101|
|2026-04-09|     191150.0|        8992|
|2026-04-10|     195412.0|        8993|
|2026-04-11|     194493.5|        8941|
+----------+-------------+------------+



##Window Function 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum as spark_sum

window_spec = Window.partitionBy("city").orderBy(col("item_revenue").desc())

df_city_item_revenue = df_silver.groupBy("city", "item_name") \
    .agg(spark_sum("total_amount").alias("item_revenue"))

df_top_item_by_city = df_city_item_revenue \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1)

df_top_item_by_city.show()

##ranking the highest-revenue item in each city using a window function

+-------------+------------+------------+----+
|         city|   item_name|item_revenue|rank|
+-------------+------------+------------+----+
|  Los Angeles|Chicken Bowl|    128500.0|   1|
|    San Diego|Chicken Bowl|    128825.0|   1|
|San Francisco|Chicken Bowl|    131875.0|   1|
+-------------+------------+------------+----+



## Insights

- The dataset was scaled to approximately 100,000 records, providing a more realistic representation of a large-scale data processing scenario.

- San Francisco generated the highest total revenue, followed by Los Angeles and San Diego, indicating strong regional performance differences.

- Salad, Smoothie, and Veg Wrap were among the most frequently ordered items, reflecting consistent customer demand across categories.

- Peak ordering activity was observed during afternoon and late evening hours, particularly between 2 PM and 4 PM, indicating key business hours for higher demand.

- Online orders had a slightly higher average order value compared to in-store orders, suggesting differences in customer behavior across order modes.

- Standard orders were slightly more common than customized orders, with both showing similar average order values, indicating balanced customer preferences.

- Daily revenue trend analysis showed consistent ordering patterns across the simulated time period, adding time-series depth to the pipeline.

- The use of a window function enabled identification of the highest revenue-generating item in each city, demonstrating advanced analytical capability.

- Overall, the pipeline successfully transformed raw transactional data into meaningful business insights using a structured lakehouse architecture and Spark-based analytics.